# ADC Construct Representation

**v0.1 educational notebook**

## Goal
Change the unit of analysis from **one target per row** to **one ADC construct per row**. This notebook is the bridge from Phase I target discovery to Phase II construct optimization.

> **SIMULATED CONSTRUCT DATA:** Component properties and outcomes in this notebook are educational synthetic values.

## Canonical guide

### Learning objectives
- Define a canonical construct-level schema.
- Separate component properties from emergent construct properties.
- Represent Target × Antibody × Linker × Payload × DAR/Conjugation interactions.

### Data mode
Fully simulated construct/component data used only to define representation and data flow.

### Inputs
None required; this bridge notebook is independently executable.

### Canonical outputs
`outputs/adc_targets_v01.csv`, `outputs/adc_antibodies_v01.csv`, `outputs/adc_linkers_v01.csv`, `outputs/adc_payloads_v01.csv`, `outputs/adc_conjugation_v01.csv`, `outputs/adc_construct_dataset_v01.csv`

### Scientific limitation
Synthetic construct outcomes are not empirical ADC measurements. Component scores and interaction formulas exist only to demonstrate representation.

### Next step
Phase II begins with multi-objective ADC construct optimization.


In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
print('ROOT:', ROOT.resolve())

ROOT: /mnt/data/audit_adc/adc-scientific-ai


In [2]:
import numpy as np
import pandas as pd
rng=np.random.default_rng(45)
targets=pd.DataFrame({'target_id':['ERBB2','TACSTD2','NECTIN4'],'tumor_score':[.90,.85,.80],'selectivity_score':[.82,.88,.91],'prevalence_score':[.70,.86,.65],'normal_risk':[.22,.18,.15]})
antibodies=pd.DataFrame({'antibody_id':['AB_001','AB_002','AB_003'],'affinity_kd_nm':[1.2,.3,4.5],'internalization_score_ab':[.80,.65,.88],'stability_score_ab':[.90,.82,.86],'aggregation_risk_ab':[.10,.18,.12]}); antibodies['affinity_log']=-np.log10(antibodies.affinity_kd_nm*1e-9)
linkers=pd.DataFrame({'linker_id':['LNK_001','LNK_002','LNK_003'],'linker_type':['protease','non-cleavable','acid'],'plasma_stability':[.82,.95,.70],'release_score':[.85,.55,.75],'hydrophobicity_risk':[.30,.20,.45]})
payloads=pd.DataFrame({'payload_id':['PAY_001','PAY_002','PAY_003'],'payload_class':['microtubule','topoisomerase','DNA_damage'],'potency_score':[.90,.95,.88],'permeability_score':[.45,.82,.60],'hydrophobicity_risk_payload':[.55,.40,.62],'bystander_score':[.40,.85,.55]})
conj=pd.DataFrame({'conjugation_id':['CONJ_001','CONJ_002','CONJ_003'],'dar_mean':[2,4,8],'dar_variance':[.1,.2,.5],'site_specific':[1,1,0],'conjugation_stability':[.95,.90,.75]})

In [3]:
design=pd.DataFrame({'construct_id':[f'ADC_{i:03d}' for i in range(1,13)],
'target_id':['ERBB2','ERBB2','ERBB2','ERBB2','TACSTD2','TACSTD2','TACSTD2','TACSTD2','NECTIN4','NECTIN4','NECTIN4','NECTIN4'],
'antibody_id':['AB_001','AB_001','AB_002','AB_003']*3,
'linker_id':['LNK_001','LNK_002','LNK_001','LNK_003']*3,
'payload_id':['PAY_001','PAY_002','PAY_003','PAY_002']*3,
'conjugation_id':['CONJ_002','CONJ_001','CONJ_003','CONJ_002']*3})
constructs=(design.merge(targets,on='target_id').merge(antibodies,on='antibody_id').merge(linkers,on='linker_id').merge(payloads,on='payload_id').merge(conj,on='conjugation_id'))
constructs['dar_optimality']=np.exp(-((constructs.dar_mean-4)**2)/8)
constructs.head()

,construct_id,target_id,antibody_id,linker_id,payload_id,conjugation_id,tumor_score,selectivity_score,prevalence_score,normal_risk,affinity_kd_nm,internalization_score_ab,stability_score_ab,aggregation_risk_ab,affinity_log,linker_type,plasma_stability,release_score,hydrophobicity_risk,payload_class,potency_score,permeability_score,hydrophobicity_risk_payload,bystander_score,dar_mean,dar_variance,site_specific,conjugation_stability,dar_optimality
0,ADC_001,ERBB2,AB_001,LNK_001,PAY_001,CONJ_002,0.90,0.82,0.70,0.22,1.2,0.80,0.90,0.10,8.920819,protease,0.82,0.85,0.30,microtubule,0.90,0.45,0.55,0.40,4,0.2,1,0.90,1.000000
1,ADC_002,ERBB2,AB_001,LNK_002,PAY_002,CONJ_001,0.90,0.82,0.70,0.22,1.2,0.80,0.90,0.10,8.920819,non-cleavable,0.95,0.55,0.20,topoisomerase,0.95,0.82,0.40,0.85,2,0.1,1,0.95,0.606531
2,ADC_003,ERBB2,AB_002,LNK_001,PAY_003,CONJ_003,0.90,0.82,0.70,0.22,0.3,0.65,0.82,0.18,9.522879,protease,0.82,0.85,0.30,DNA_damage,0.88,0.60,0.62,0.55,8,0.5,0,0.75,0.135335
3,ADC_004,ERBB2,AB_003,LNK_003,PAY_002,CONJ_002,0.90,0.82,0.70,0.22,4.5,0.88,0.86,0.12,8.346787,acid,0.70,0.75,0.45,topoisomerase,0.95,0.82,0.40,0.85,4,0.2,1,0.90,1.000000
4,ADC_005,TACSTD2,AB_001,LNK_001,PAY_001,CONJ_002,0.85,0.88,0.86,0.18,1.2,0.80,0.90,0.10,8.920819,protease,0.82,0.85,0.30,microtubule,0.90,0.45,0.55,0.40,4,0.2,1,0.90,1.000000


In [4]:
constructs['binding_score']=np.clip(.45*constructs.tumor_score+.35*constructs.internalization_score_ab+.20*rng.random(len(constructs)),0,1)
constructs['internalization_score']=np.clip(.55*constructs.internalization_score_ab+.20*constructs.binding_score+.25*rng.random(len(constructs)),0,1)
constructs['payload_release_score']=np.clip(.50*constructs.release_score+.25*constructs.plasma_stability+.25*rng.random(len(constructs)),0,1)
constructs['aggregation_risk']=np.clip(.25*constructs.aggregation_risk_ab+.25*constructs.hydrophobicity_risk+.25*constructs.hydrophobicity_risk_payload+.25*(constructs.dar_mean/8),0,1)
constructs['cytotoxicity_score']=np.clip(.20*constructs.binding_score+.25*constructs.internalization_score+.20*constructs.payload_release_score+.20*constructs.potency_score+.15*constructs.dar_optimality,0,1)
constructs['bystander_heterogeneity_benefit']=constructs.bystander_score*(1-constructs.prevalence_score)
constructs['normal_toxicity_score']=np.clip(.40*constructs.normal_risk+.25*constructs.permeability_score+.20*constructs.hydrophobicity_risk_payload+.15*(constructs.dar_mean/8),0,1)
constructs['adc_plasma_stability']=np.clip(.50*constructs.plasma_stability+.30*constructs.conjugation_stability-.20*constructs.aggregation_risk,0,1)
constructs['functional_adc_score']=np.clip(.65*constructs.cytotoxicity_score+.15*constructs.bystander_heterogeneity_benefit+.20*constructs.conjugation_stability-.30*constructs.aggregation_risk-.30*constructs.normal_toxicity_score,0,1)
constructs.sort_values('functional_adc_score',ascending=False)[['construct_id','target_id','antibody_id','linker_id','payload_id','dar_mean','cytotoxicity_score','normal_toxicity_score','functional_adc_score']]

,construct_id,target_id,antibody_id,linker_id,payload_id,dar_mean,cytotoxicity_score,normal_toxicity_score,functional_adc_score
11,ADC_012,NECTIN4,AB_003,LNK_003,PAY_002,4,0.830779,0.4200,0.528381
0,ADC_001,ERBB2,AB_001,LNK_001,PAY_001,4,0.846177,0.3855,0.523615
9,ADC_010,NECTIN4,AB_001,LNK_002,PAY_002,2,0.729630,0.3825,0.522884
1,ADC_002,ERBB2,AB_001,LNK_002,PAY_002,2,0.745212,0.4105,0.518238
5,ADC_006,TACSTD2,AB_001,LNK_002,PAY_002,2,0.758421,0.3945,0.511224
3,ADC_004,ERBB2,AB_003,LNK_003,PAY_002,4,0.826828,0.4480,0.511038
4,ADC_005,TACSTD2,AB_001,LNK_001,PAY_001,4,0.813149,0.3695,0.497347
8,ADC_009,NECTIN4,AB_001,LNK_001,PAY_001,4,0.785346,0.3575,0.495475
7,ADC_008,TACSTD2,AB_003,LNK_003,PAY_002,4,0.785024,0.4320,0.468265
2,ADC_003,ERBB2,AB_002,LNK_001,PAY_003,8,0.686706,0.5120,0.310009


In [5]:
# Component tables remain normalized; the construct dataset is the ML-ready bridge.
targets.to_csv(OUTPUTS/'adc_targets_v01.csv',index=False)
antibodies.to_csv(OUTPUTS/'adc_antibodies_v01.csv',index=False)
linkers.to_csv(OUTPUTS/'adc_linkers_v01.csv',index=False)
payloads.to_csv(OUTPUTS/'adc_payloads_v01.csv',index=False)
conj.to_csv(OUTPUTS/'adc_conjugation_v01.csv',index=False)
constructs.to_csv(OUTPUTS/'adc_construct_dataset_v01.csv',index=False)
assert constructs.construct_id.is_unique
print('saved construct dataset:',len(constructs),'rows')

saved construct dataset: 12 rows


## Phase transition
Notebooks 01–09 ask **Which target should we prioritize and test?** Notebook 10 changes the question to **Which combination of target, antibody, linker, payload, DAR, and conjugation should we develop?** This creates the combinatorial design space required for multi-objective optimization in Phase II.